# Messages

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM. Messages are objects that contain:
- Role - Identifies the message type (e.g. system, user)
- Content - Represents the actual content f the message (like text, images, audio, documents, etc.)
- Metadata - Optional fields such as response information, message IDs, and token usage

LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")

model.invoke("Please tell me what is apple inteligince?")

AIMessage(content='<think>\nOkay, the user is asking about "apple inteligence." First, I need to figure out what they\'re referring to. Since there\'s a typo, maybe they meant "Apple Intelligence." I should check if Apple has something called Apple Intelligence. \n\nI recall that Apple has been working on machine learning and AI features across their products. For example, they have things like Apple Intelligence in iOS 18, which includes features like Smart Reply, Mail Drop, and enhancements in Safari. Maybe the user is asking about that.\n\nWait, but I should confirm the correct name. Apple doesn\'t use the term "Apple Intelligence" officially. They usually refer to their AI and machine learning capabilities as part of their ecosystem. Maybe the user is confusing it with another company\'s product. For example, Google has Google Assistant and Google AI, Microsoft has Azure AI, etc.\n\nAlternatively, maybe the user is referring to Apple\'s use of AI in their products like Siri, Core M

## Text Prompts

Text prompts are strings - ideal for straightforward generation tasks where you don't need to retain conversation history.

In [ ]:
model.invoke("What is langchain ?")

Use text prompts when:
- You have a single, standalone request
- You don't need conversation history
- You want minimal code complexity

## Message Prompts
Alternatively, you can pass in a list of messages to the model by providing a list of message objects.

Message types
- System message - Tells the model how to behave and provide context for interactions
- Human message - Represents user input and interactions with the model
- Al message - Responses generated by the model, including text content, tool calls, and metadata
- Tool message - Represents the outputs of tool calls

In [2]:
from langchain.messages import SystemMessage,HumanMessage,AIMessage

# messages=[
#     SystemMessage("You are a famous poet from India"),
#     HumanMessage("Write a poem on love")
# ]
# response=model.invoke(messages)

system_msg=SystemMessage("""
You are a senior Python developer with expertise in web frameworks.
Always provide code examples and explain your reasoning.
Be concise but thorough in your explanations.
""")

messages=[
    system_msg,
    HumanMessage("How do i create a REST API?")
]
response=model.invoke(messages)
print(response)
print(response.content)
print(response.text)

content='<think>\nOkay, the user wants to create a REST API. Let me think about the best way to approach this. They\'re probably looking for a step-by-step guide with code examples. Since they mentioned being a senior Python developer, maybe they have some experience but need a refresher or a specific example.\n\nFirst, I should choose a framework. Flask and FastAPI are popular in Python. FastAPI is modern, has async support, and automatic documentation. Maybe I\'ll go with FastAPI because it\'s efficient and user-friendly.\n\nNext, I need to outline the steps. Install FastAPI and an ASGI server like Uvicorn. Then set up a basic app with routes. Include CRUD operations as examples since that\'s common in REST APIs. Maybe use an in-memory database for simplicity, but mention that a real database would be used in production.\n\nI should explain each part of the code. For instance, the POST, GET, PUT, and DELETE methods. Also, include how to run the server and test the API. Don\'t forget 

In [3]:
print(response.text)

<think>
Okay, the user wants to create a REST API. Let me think about the best way to approach this. They're probably looking for a step-by-step guide with code examples. Since they mentioned being a senior Python developer, maybe they have some experience but need a refresher or a specific example.

First, I should choose a framework. Flask and FastAPI are popular in Python. FastAPI is modern, has async support, and automatic documentation. Maybe I'll go with FastAPI because it's efficient and user-friendly.

Next, I need to outline the steps. Install FastAPI and an ASGI server like Uvicorn. Then set up a basic app with routes. Include CRUD operations as examples since that's common in REST APIs. Maybe use an in-memory database for simplicity, but mention that a real database would be used in production.

I should explain each part of the code. For instance, the POST, GET, PUT, and DELETE methods. Also, include how to run the server and test the API. Don't forget to mention the automa

In [5]:
# Message metadata

human_msg=HumanMessage(
    content="Hello",
    name="Jane", # optional : identify different users
    id="msg_123" # optional : unique identifier for tracing
)

response=model.invoke([human_msg])
response

AIMessage(content='<think>\nOkay, the user sent "Hello". That\'s pretty straightforward. I should respond in a friendly and welcoming manner. Let me check the guidelines to make sure I\'m on track. They want the response to be natural and conversational. I need to avoid any markdown and keep it simple. Maybe start with a greeting and offer help. Let me see... "Hello! How can I assist you today?" That sounds good. It\'s open-ended and invites them to ask for help. I should keep it like that. No need for extra fluff. Just make sure it\'s warm and approachable.\n</think>\n\nHello! How can I assist you today? 😊', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 137, 'prompt_tokens': 9, 'total_tokens': 146, 'completion_time': 0.287695542, 'completion_tokens_details': None, 'prompt_time': 0.000576489, 'prompt_tokens_details': None, 'queue_time': 0.193204462, 'total_time': 0.288272031}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'servic

In [ ]:
# create an AI message manually (e.g for conversation history)
ai_msg=AIMessage("I'd happy to help you with this question")

# Add to the converstaion history
messages=[
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,
    HumanMessage("What's 3+2?")
]
response=model.invoke(messages)
print(response.content)

In [6]:
response.usage_metadata

{'input_tokens': 9, 'output_tokens': 137, 'total_tokens': 146}

In [12]:
from langchain.messages import ToolMessage

ai_message=AIMessage(
    content=[],
    tool_calls=[{
        "name":"get_weather",
        "args":{"location":"San Francisco"},
        "id": "call_12321"
    }]
)

weather_result="Sunny, 72 F"
tool_msg=ToolMessage(
    content=weather_result,
    tool_call_id="call_12321" # must match the call id
)

messages=[
    HumanMessage("What's the weather in san francisco?"),
    ai_message, # model's tool call
    tool_msg # Tool execution 
]

response=model.invoke(messages)
response

AIMessage(content='<think>\nOkay, the user asked for the weather in San Francisco. I used the get_weather function and got back "Sunny, 72 F". Now I need to present this information clearly. Let me check if there\'s anything else the user might need. Maybe they want to know about the day or time? The current response is straightforward, but adding a bit more context could be helpful. Let me make sure the temperature is in the right unit and the condition is correctly mentioned. Alright, the response looks good. Just need to format it in a friendly and concise way.\n</think>\n\nThe current weather in San Francisco is **sunny** with a temperature of **72°F**. Perfect weather for a walk or outdoor activities! ☀️', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 154, 'prompt_tokens': 58, 'total_tokens': 212, 'completion_time': 0.328796785, 'completion_tokens_details': None, 'prompt_time': 0.005357705, 'prompt_tokens_details': None, 'queue_time': 0.158671779, '

```Python
ai_message=AIMessage( 
    content=[], 
    tool_calls=[
        { 
            "name":"get_weather", 
            "args":{"vbcvb":"Franvhvm cisco"}, 
            "id": "call_12321" 
        }
    ]) 
    weather_result="Sunny, 72 F" 
    tool_msg=ToolMessage( 
        content=weather_result, 
        tool_call_id="call_1321" # must match the call id 
    ) 
    messages=[ 
        HumanMessage("What's the weather in san francisco?"), 
        ai_message, # model's tool call 
        tool_msg # Tool execution 
    ] 
    response=model.invoke(messages) 
    response
```
# Why Did This Work Even Though Everything Was Wrong?

In this example, there was:
- no real `get_weather` execution
- invalid tool arguments
- mismatched `tool_call_id`
- fake tool output

yet the model still generated a correct-looking answer.

---

## Important Concept

LLMs do not actually validate or execute tools.

They simply read the conversation history and predict the next most likely response.

The model trusted the messages we manually created.

---

## Why Invalid Arguments Did Not Matter

We passed:

```python
"args":{"vbcvb":"Franvhvm cisco"}
```

which is completely invalid.

However, the model ignored these arguments because it never executed the tool.

The arguments are mainly for:
- Python systems
- agent frameworks
- external tool execution

not for the LLM's internal reasoning.

---

## Where Did "San Francisco" Come From?

The model got it from the original user message:

```python
HumanMessage("What's the weather in san francisco?")
```

The LLM relied more on the natural language conversation context than on the fake tool arguments.

So internally the model understood:

```text
User asked about San Francisco weather
↓
Tool returned weather result
↓
Generate final answer
```

---

## How Did It Know "72°F"?

Because we manually gave the tool output:

```python
ToolMessage(content="Sunny, 72 F")
```

The model simply reused that information in a natural sentence.

The LLM trusted the tool output as valid information.

It did not verify whether the temperature was real.

---

## Key Insight

The model only sees structured conversation history like:

```text
Human:
What's the weather in San Francisco?

Assistant:
Calling weather tool...

Tool:
Sunny, 72 F
```

and then predicts the next response:

```text
The weather in San Francisco is sunny with 72°F.
```

---

# Final Takeaway

LLMs are not executing programs internally.

They are:
- reading context
- identifying patterns
- trusting conversation history
- predicting the next response

Tool execution and validation happen externally through Python or agent frameworks.